## Pillar 1: Iterators, Generators & Streaming Pipelines

Core Mechanics
1. The Iterator Protocol Under the Hood
In Python, any object is an iterable if it implements __iter__(). It becomes an iterator when it also implements __next__().

iter(obj) calls obj.__iter__(), which returns the iterator object itself.

next(it) calls it.__next__(), which returns the next item or raises StopIteration when exhausted.

Iterable: Any object that implements __iter__(), which must return an iterator instance. Examples: list, dict, set, str.

When training an RL agent or loading a giant dataset, what happens if you load 10 million states/transitions into a standard Python list?

Python
# BAD: Allocates 10 million items in RAM at once (Crashes your memory!)
data = [process_step(i) for i in range(10_000_000)]

In [ ]:
class CountDown:
    def __init__(self, start):
        self.curr = start
    def __iter__(self):
        return self
    def __next__(self):
        if self.curr <= 0:
            raise StopIteration
        val = self.curr
        self.curr -= 1
        return val

1. The Core Trick: yield vs. return
return: Calculates the value, kills the function, throws away its local variables, and gives back the result.

yield: Hands you a single value, pauses the function right there, freezes its local variables in memory, and waits until you ask for the next one.

In [2]:
def simple_stream():
    
    print("first chunk")
    yield 1
    
    print("next")
    yield 2
gen =simple_stream()

print(next(gen))

first chunk
1


In [3]:
print(next(gen))

next
2


[ Iterable Object ] (e.g. list)
               │
               ▼  iter(obj) -> calls obj.__iter__()
       [ Iterator Object ] (Holds pointer / cursor state)
               │
               ▼  next(it) -> calls it.__next__()
         [ Value 1 ]
               │
               ▼  next(it)
         [ Value 2 ]
               │
               ▼  next(it)
        StopIteration Exception (Loop catches and terminates cleanly)

what python does in loops
# What you write:
for x in collection:
    process(x)

# What CPython actually executes:
_iter = iter(collection)         # _iter = collection.__iter__()
while True:
    try:
        x = next(_iter)          # x = _iter.__next__()
    except StopIteration:
        break
    process(x)

2. Generators as State Machines
A generator function is a concise way to create an iterator without manually writing a class with __iter__ and __next__.

When a function contains the yield keyword, calling it does not execute the body; it compiles to a PyGenObject.

When next(gen) is called, the CPython evaluation loop executes bytecodes until it encounters a YIELD_VALUE opcode.

The interpreter pauses the frame, saves its instruction pointer (f_lasti) and local variables (f_locals), and returns the yielded value.

The next next(gen) call restores the exact frame context and resumes execution immediately after the yield.

In [4]:
def count_upto(num):
    current = 1
    while current <= num:
        yield current
        current+=1

gen = count_upto(3)
print(next(gen))

1


In [5]:
print(next(gen))

2


In [6]:
print(next(gen))

3


In [7]:
print(next(gen))

StopIteration: 

Two-Way Communication (.send() and .throw())
Standard generators only push values out (via yield). However, Python generators can also act as coroutines, meaning you can send data back into the generator while it is paused.

1. .send(value): Pushing data into a paused generator
When you write received = yield output_val:

yield output_val sends output_val out to whoever called next().

The generator pauses.

When someone calls gen.send(new_val), new_val is assigned to the received variable on the left, and execution resumes.

In [17]:
def gen_with_comms():
    total = 0
    while True:
        increment = yield total
        if increment is not None:
            total+=increment

acc = gen_with_comms()
print(next(acc))

0


In [18]:
print(next(acc))

0


In [19]:
print(acc.send(2))

2


In [20]:
print(acc.send(2))

4


2. .throw(ExceptionType): Injecting errors into a generator
Instead of sending data, .throw() raises an exception inside the generator at the exact line where it is currently paused. This is useful for resetting state or triggering graceful shutdowns.

In [21]:
def worker():
    try:
        while True:
            yield "Working..."
    except CustomResetError:
        print("Received reset signal! Cleaning up...")
        yield "Reset complete"

class CustomResetError(Exception):
    pass

w = worker()
print(next(w))  # Outputs: "Working..."

# Inject an exception into the paused generator
print(w.throw(CustomResetError))  # Triggers the 'except' block -> outputs "Reset complete"

Working...
Received reset signal! Cleaning up...
Reset complete


In [22]:
def sub_routine():
    
    yield "A from sub"
    
    yield "B from sub"
    
def main_routine():
    yield "main started"
    
    yield from sub_routine()
    
    yield "main ended"

for a in main_routine():
    print(a)

main started
A from sub
B from sub
main ended


## Building an Event Stream Step-by-Step